In [1]:
import torch
from pyqcu import tools, dslash, lattice
from pyqcu.cuda import qcu, define
from pyqcu.cuda.define import params, argv, set_ptrs
params[define._LAT_X_] = 4
params[define._LAT_Y_] = 4
params[define._LAT_Z_] = 4
params[define._LAT_T_] = 8
params[define._LAT_XYZT_] = params[define._LAT_X_] * \
    params[define._LAT_Y_]*params[define._LAT_Z_]*params[define._LAT_T_]
params[define._GRID_X_], params[define._GRID_Y_], params[define._GRID_Z_], params[
    define._GRID_T_] = tools.give_grid_size()
params[define._PARITY_] = 0
params[define._NODE_RANK_] = define.rank
params[define._NODE_SIZE_] = define.size
params[define._DAGGER_] = 0
params[define._MAX_ITER_] = 1000
params[define._DATA_TYPE_] = define._LAT_C64_
# params[define._DATA_TYPE_] = define._LAT_C128_
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = 1
params[define._MG_X_] = 1
params[define._MG_Y_] = 1
params[define._MG_Z_] = 1
params[define._MG_T_] = 1
params[define._LAT_E_] = 24
params[define._VERBOSE_] = 1
params[define._SEED_] = 42
argv = argv.to(dtype=define.dtype(params[define._DATA_TYPE_]).to_real())
argv[define._MASS_] = 0.05
argv[define._TOL_] = 1e-9
argv[define._SIGMA_] = 0.1
print(params)
print(argv)
print(set_ptrs)
gauge_eo = torch.zeros(size=[2, 3, 3, 4]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                       params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                            params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.rand_like(fermion_in_eo)
fermion_out_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                             params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([   4,    4,    4,    8,  512,    1,    1,    1,    1,    0,    0,    1,
           0, 1000,    2,    0,    1,    1,    1,    1,    1,   24,    1,   42,
           0], dtype=torch.int32)
tensor([5.0000e-02, 1.0000e-09, 1.0000e-01])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [ ]:
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = -1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyGaussGaugeQcu(gauge_eo, set_ptrs, params)

print(lattice.check_su3(U=gauge_eo[0]))
print(lattice.check_su3(U=gauge_eo[1]))

set_ptr:0x55e3b8744ce0
set_ptrs:0x55e3b73fb3c0
long long set_ptr:94436540566752
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :0
host_params[_SET_PLAN_]      :-1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
hos

In [3]:
_gauge_eo=gauge_eo.clone()
_fermion_in_eo=fermion_in_eo.clone()
_fermion_out_eo=fermion_out_eo.clone()

In [4]:
print('Difference:', tools.norm(gauge_eo-_gauge_eo)/tools.norm(_gauge_eo))
print('Difference:', tools.norm(fermion_in_eo-_fermion_in_eo)/tools.norm(_fermion_in_eo))

Difference: 0.0
Difference: 0.0


In [ ]:

print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonBistabCgQcu(
    fermion_out_eo, fermion_in_eo, gauge_eo, set_ptrs, params)
# 
print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))


tensor([94436540566752,              0,              0,              0,
                     0,              0,              0,              0,
                     0,              0])
set_ptr:0x55e3c2de1570
set_ptrs:0x55e3b73fb3c0
long long set_ptr:94436715271536
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :1
host_params[_SET_PLAN_]      :1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]     

In [ ]:

clover_ee = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_ee_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
# gauge_eo = torch.ones_like(gauge_eo)*2
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_ee, clover_ee_inv, gauge_eo, set_ptrs, params)

print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_oo, clover_oo_inv, gauge_eo, set_ptrs, params)

print(set_ptrs)
fermion_out_eo = torch.zeros_like(fermion_out_eo)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverBistabCgQcu(fermion_out_eo, fermion_in_eo, gauge_eo,
                           clover_ee, clover_oo, clover_ee_inv, clover_oo_inv,  set_ptrs, params)

print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)+dslash.give_clover(src=qcu_dest, clover_term=refer_clover_term)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))

set_ptr:0x55e3c5bc7660
set_ptrs:0x55e3b73fb3c0
long long set_ptr:94436763399776
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :2
host_params[_SET_PLAN_]      :2
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [7]:

clover_eeoo = torch.zeros(size=[2, 4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                                params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_eeoo[0] = clover_ee
clover_eeoo[1] = clover_oo
qcu_clover_term = tools.poooxyzt2oooxyzt(
    input_array=clover_eeoo)
qcu_clover_term = dslash.cut_I(clover_term=qcu_clover_term)

qcu_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=qcu_clover_term)
refer_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term)
print('qcu_clover_term:', qcu_clover_term.flatten()[:100])
print('refer_clover_term:', refer_clover_term.flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[0].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[0].flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[1].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[1].flatten()[:100])
print('Difference:', tools.norm(refer_clover_term -
      qcu_clover_term)/tools.norm(qcu_clover_term))


qcu_clover_term: tensor([-1.5640e-03+0.j, -1.2131e-03+0.j,  1.5091e-03+0.j, -1.3459e-03+0.j,
        -1.7273e-04+0.j, -1.2975e-03+0.j, -4.6636e-03+0.j,  7.6789e-03+0.j,
        -3.5279e-03+0.j, -3.5028e-03+0.j,  3.3808e-04+0.j,  4.6502e-03+0.j,
         4.1162e-03+0.j, -2.6107e-04+0.j, -8.7095e-03+0.j, -3.6175e-03+0.j,
        -2.7232e-03+0.j,  4.7187e-03+0.j, -9.7674e-04+0.j,  4.6992e-04+0.j,
         7.1980e-03+0.j, -2.6121e-03+0.j, -1.0670e-03+0.j, -7.8315e-04+0.j,
         3.3987e-03+0.j, -3.1114e-03+0.j,  7.3184e-03+0.j,  3.5887e-03+0.j,
         1.0196e-03+0.j,  5.3486e-03+0.j,  1.0300e-04+0.j,  1.2250e-02+0.j,
        -8.3512e-04+0.j,  2.9544e-03+0.j, -1.1584e-03+0.j, -3.8020e-03+0.j,
         2.5309e-03+0.j, -5.6899e-03+0.j,  1.6824e-03+0.j,  1.9593e-03+0.j,
        -4.5556e-03+0.j, -2.5800e-03+0.j,  6.6400e-04+0.j, -1.4204e-03+0.j,
         2.4658e-03+0.j,  1.7161e-03+0.j, -2.2391e-03+0.j, -1.8537e-04+0.j,
        -4.3339e-04+0.j, -1.9675e-03+0.j,  3.3346e-03+0.j,  2.2736e-03+

In [8]:
print('qcu_dest:', qcu_dest.flatten()[:100])

qcu_dest: tensor([11.6005+4.9855j, 10.8202+5.1332j, 10.5166+4.9331j, 11.1391+5.8649j,
        11.7919+5.0730j, 11.1646+5.3420j, 10.7787+4.8990j, 11.0169+4.0963j,
        12.2095+5.3348j, 11.4886+5.0876j, 11.6934+5.2559j, 11.6055+4.6871j,
        12.7933+4.0226j, 12.2308+5.0379j, 12.3880+5.0809j, 12.2451+4.7311j,
        12.2754+5.1097j, 11.6903+5.7388j, 12.2535+5.8203j, 11.6417+5.5291j,
        12.2761+4.4325j, 12.2226+6.0089j, 12.2196+4.1245j, 11.5758+5.0739j,
        12.1688+4.1888j, 11.5113+4.7682j, 11.7541+4.5949j, 11.5584+4.5126j,
        12.4852+4.4117j, 11.5991+4.8404j, 11.5109+4.0017j, 10.8596+4.4363j,
        13.1068+4.1000j, 12.4585+5.6966j, 11.9022+5.7063j, 12.7418+5.6164j,
        11.5486+4.0018j, 11.7079+4.5125j, 12.1195+5.1386j, 11.7974+4.0332j,
        13.2149+5.0169j, 13.1638+5.4627j, 12.4630+5.1558j, 13.8299+4.9031j,
        13.0534+3.5132j, 12.3954+3.8931j, 11.9689+4.2273j, 12.5541+5.1840j,
        13.7902+6.1905j, 13.7210+5.5801j, 13.1778+5.7623j, 13.6304+5.6002j,
  

In [9]:
a = qcu_src-refer_src
print('a:', a.flatten()[:100])

a: tensor([ 5.0911e-01-0.1519j,  5.6399e-01-0.1727j,  3.2593e-01-0.1554j,
         2.6068e-01-0.3062j,  3.7611e-01-0.1703j,  2.0192e-01-0.1380j,
         2.6530e-01-0.2598j,  2.3802e-01-0.1671j,  2.5614e-01-0.0780j,
         2.1509e-01-0.2248j,  2.1772e-01-0.1660j,  4.5411e-01-0.2364j,
         2.9650e-01-0.3304j,  1.9977e-01-0.1460j,  2.1431e-01-0.1797j,
         3.3207e-01-0.0798j,  4.5890e-01+0.0417j,  2.1546e-01-0.2242j,
         2.6563e-01-0.2264j,  3.2104e-01-0.0797j,  1.5204e-01-0.2196j,
         3.9820e-01-0.1521j,  3.0752e-01-0.1805j,  1.2526e-01-0.1978j,
         1.9453e-01-0.0566j,  1.1904e-01-0.1811j,  2.2590e-01-0.0565j,
         1.8584e-01-0.0161j,  4.6005e-02-0.1451j,  3.7674e-01-0.1105j,
         1.4163e-01-0.1999j,  5.3364e-02+0.0668j,  1.7122e-01-0.4361j,
         1.1346e-01-0.4429j,  3.8140e-03-0.3516j,  1.3024e-01-0.2126j,
         1.8119e-01-0.2230j, -1.8077e-02-0.3269j,  1.2887e-01-0.3470j,
         1.0475e-02-0.4099j,  1.1215e-01-0.2163j,  9.0324e-02-0.1926j,
   

In [10]:
qcu_src

tensor([[[[[[4.6818e-01+2.8463e-01j, 3.5071e-01+1.9987e-01j,
             9.2243e-03+6.9233e-01j,  ...,
             3.5133e-01+6.2948e-01j, 3.5228e-01+5.2612e-01j,
             2.9508e-01+3.3406e-01j],
            [8.3663e-01+6.5238e-01j, 1.7409e-01+6.8482e-01j,
             5.9401e-01+7.6559e-01j,  ...,
             9.0269e-01+8.8137e-01j, 9.1316e-01+6.3108e-01j,
             5.5772e-01+4.3105e-02j],
            [7.1671e-01+9.4219e-01j, 1.7115e-01+8.7114e-01j,
             7.6539e-01+5.9411e-01j,  ...,
             3.9616e-01+8.8650e-01j, 7.3857e-01+1.8259e-01j,
             1.8029e-01+6.5823e-01j],
            [4.7614e-01+4.4857e-01j, 1.8040e-01+3.6577e-01j,
             8.2658e-01+9.8221e-01j,  ...,
             1.9987e-01+5.5858e-01j, 9.6002e-01+4.5275e-01j,
             1.0258e-01+8.5250e-01j]],

           [[9.1858e-01+2.5159e-01j, 6.3686e-01+8.9744e-01j,
             4.6527e-01+8.0254e-01j,  ...,
             6.6917e-01+1.4654e-01j, 9.4093e-01+8.6288e-01j,
             2.5974e-

In [11]:
refer_src

tensor([[[[[[-4.0936e-02+4.3658e-01j, -2.1329e-01+3.7260e-01j,
             -3.1671e-01+8.4774e-01j,  ...,
              1.4941e-01+7.6746e-01j,  8.6972e-02+7.8587e-01j,
              5.7064e-02+5.0112e-01j],
            [ 5.8050e-01+7.3034e-01j, -4.0995e-02+9.0957e-01j,
              3.7629e-01+9.3156e-01j,  ...,
              7.0292e-01+1.0274e+00j,  6.9885e-01+8.1083e-01j,
              2.2565e-01+1.2286e-01j],
            [ 2.5781e-01+9.0050e-01j, -4.4310e-02+1.0953e+00j,
              4.9975e-01+8.2055e-01j,  ...,
             -2.0386e-03+1.0386e+00j,  4.3104e-01+3.6313e-01j,
              5.5031e-02+8.5605e-01j],
            [ 2.8161e-01+5.0520e-01j,  6.1362e-02+5.4684e-01j,
              6.0068e-01+1.0387e+00j,  ...,
             -1.7687e-01+6.6910e-01j,  8.1839e-01+6.5267e-01j,
              4.9213e-02+7.8567e-01j]],

           [[ 7.4736e-01+6.8765e-01j,  5.2339e-01+1.3403e+00j,
              4.6146e-01+1.1542e+00j,  ...,
              6.8725e-01+4.7347e-01j,  8.1206e-01+1.209

In [ ]:
fermion_out_eo = torch.zeros_like(fermion_out_eo)
fermion_in_o=fermion_in_eo[1]
fermion_out_e=fermion_out_eo[0]
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverDslashQcu(fermion_out_e, fermion_in_o, gauge_eo, set_ptrs, params)

refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_clover_term = dslash.add_I(clover_term=refer_clover_term)
refer_clover_term_inv = dslash.inverse(clover_term=refer_clover_term)
refer_clover_term_inv_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term_inv)
refer_clover_term_inv_e = refer_clover_term_inv_eo[0]
refer_fermion_out_e = dslash.give_clover_ee(src_e=dslash.give_wilson_eo(
    src_o=fermion_in_o, U_eo=gauge_eo, kappa=1 / (2 * argv[define._MASS_] + 8)),clover_e=refer_clover_term_inv_e)
print(tools.norm(clover_ee_inv-refer_clover_term_inv_e)/tools.norm(clover_ee_inv))
print(tools.norm(fermion_out_e-refer_fermion_out_e)/tools.norm(fermion_out_e))

set_ptr:0x55e3c7cfe540
set_ptrs:0x55e3b73fb3c0
long long set_ptr:94436798227776
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :1
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :5
host_params[_SET_PLAN_]      :2
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

In [ ]:
fermion_out_eo = torch.zeros_like(fermion_out_eo)
fermion_in_o=fermion_in_eo[1]
fermion_out_e=fermion_out_eo[0]
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonDslashQcu(fermion_out_e, fermion_in_o, gauge_eo, set_ptrs, params)

refer_fermion_out_e = dslash.give_wilson_eo(
    src_o=fermion_in_o, U_eo=gauge_eo, kappa=1 / (2 * argv[define._MASS_] + 8))
print(fermion_out_e.flatten()[:100])
print(refer_fermion_out_e.flatten()[:100])
print(tools.norm(fermion_out_e-refer_fermion_out_e)/tools.norm(fermion_out_e))

set_ptr:0x55e3c9233b90
set_ptrs:0x55e3b73fb3c0
long long set_ptr:94436820466576
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :1
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :6
host_params[_SET_PLAN_]      :1
host_params[_MG_X_]          :1
host_params[_MG_Y_]          :1
host_params[_MG_Z_]          :1
host_params[_MG_T_]          :1
host_params[_LAT_E_]         :24
host_params[_VERBOSE_]       :1
host_params[_SEED_]          :42
host_params[_TEST_IN_CPU_]   :0
host_argv[_MASS_]            :5.000000e-02
host

tensor([3.2756+6.5207j, 3.9359+6.5846j, 4.7542+6.0102j, 6.9341+6.0912j,
        4.5360+6.0593j, 3.8849+5.2573j, 4.8945+6.4930j, 7.3980+3.6554j,
        2.7848+6.0889j, 5.3456+5.2298j, 5.2710+5.3871j, 2.6619+4.5930j,
        5.6878+5.1830j, 4.7644+4.5477j, 5.2808+6.3741j, 2.3522+6.7106j,
        4.9594+3.6826j, 4.0494+6.2580j, 5.1631+3.6104j, 2.3726+4.5532j,
        5.7721+5.5198j, 5.2199+4.5143j, 4.1071+1.9316j, 3.2444+4.4135j,
        4.3038+5.6805j, 4.0287+4.4949j, 5.6143+3.3893j, 4.4720+4.4590j,
        6.8467+2.3035j, 5.2695+4.6952j, 4.2784+2.6480j, 6.5145+5.5365j,
        3.0858+4.0593j, 3.1719+5.5621j, 2.8810+2.2688j, 4.1094+4.1573j,
        3.8087+4.1599j, 5.2133+6.1223j, 3.6489+2.2337j, 3.5234+4.8368j,
        3.5568+4.7355j, 3.8414+6.3634j, 4.8176+4.0869j, 3.5234+4.5339j,
        6.6007+1.9895j, 3.8652+5.4401j, 4.5453+3.9157j, 7.3629+1.4285j,
        3.7605+5.8422j, 3.0872+5.2010j, 4.2439+3.7753j, 6.0758+4.1565j,
        7.1453+5.0828j, 5.3782+5.2985j, 3.9608+5.0844j, 5.4884+3

In [14]:
print("gauge_eo.is_contiguous():",gauge_eo.is_contiguous())
print("fermion_in_eo.is_contiguous():",fermion_in_eo.is_contiguous())
print("fermion_in_out.is_contiguous():",fermion_out_eo.is_contiguous())
print("qcu_src.is_contiguous():",qcu_src.is_contiguous())
print("qcu_dest.is_contiguous():",qcu_dest.is_contiguous())

gauge_eo.is_contiguous(): True
fermion_in_eo.is_contiguous(): True
fermion_in_out.is_contiguous(): True
qcu_src.is_contiguous(): True
qcu_dest.is_contiguous(): True
